# 🧪 Feature Pipeline Exploration & Extraction

Notebook này được sử dụng để kiểm tra luồng `build_features` của dự án Optiver. Nó thực thi master pipeline, xử lý dữ liệu raw parquet thành một DataFrame duy nhất chứa toàn bộ các features (Base + Nearest Neighbors) và lưu xuống file dạng Feather để sử dụng trong quá trình Training.

### 1. Import Thư Viện & Setup

In [ ]:
import os
import sys
import warnings
import pandas as pd
import numpy as np

# Tắt cảnh báo chia cho NaN hoặc All-NaN slice của Numpy
warnings.filterwarnings('ignore')

# Trỏ đường dẫn gốc về thư mục root dự án để import `src`
sys.path.append(os.path.abspath('..'))

from src.data.data_loader import DataBlock
from src.features.feature_pipeline import build_features

### 2. Load Base Train DataFrame
Đọc tập `train.csv` chứa `time_id`, `stock_id` và nhãn `target` (Realized Volatility tương lai).

In [ ]:
DATA_DIR = '../data'
RAW_DIR = os.path.join(DATA_DIR, 'raw/optiver-realized-volatility-prediction')
TRAIN_CSV = os.path.join(RAW_DIR, 'train.csv')

print("Đang đọc dữ liệu train.csv...")
try:
    df_train = pd.read_csv(TRAIN_CSV)
    print(f"Kích thước ban đầu: {df_train.shape}")
    display(df_train.head())
except FileNotFoundError:
    print(f"LỖI: Chưa tìm thấy file {TRAIN_CSV}. Bạn cần tải và giải nén dữ liệu Kaggle vào thư mục này.")
    # Dữ liệu giả định (Dummy Data) nếu chạy thử mà chưa có raw data
    print("Sử dụng Dummy Data để giả lập...")
    df_train = pd.DataFrame({'stock_id': [0, 0, 1, 1], 'time_id': [5, 11, 5, 11], 'target': [0.004, 0.001, 0.002, 0.003]})

### 3. Thực Thi Master Pipeline
Đoạn mã này sẽ kích hoạt `build_features` để tự động load Book, Trade Parquet files, xử lý song song và sinh ra hàng trăm đặc trưng (Feature Engineering).

In [ ]:
print("Bắt đầu khởi chạy Feature Pipeline...")

# Lưu ý: Mặc định sẽ sử dụng tất cả CPU core để tính toán.
# Việc tính K-NN Neighbors mất khá nhiều RAM. Hàm đã tối ưu bộ nhớ.
df_features = build_features(
    df_train=df_train,
    data_dir=RAW_DIR,
    block=DataBlock.TRAIN,
    config_path='../configs/feature_config.yaml'
)

### 4. Kiểm Tra & Đánh Giá Dữ Liệu Đầu Ra

In [ ]:
print(f"Kích thước DataFrame cuối cùng (N_rows, N_features): {df_features.shape}")

# Đếm số lượng NaN. Sẽ luôn có một chút NaN do K-NN tìm Neighbors không đủ dữ liệu
# LightGBM xử lý được NaN nên không cần drop/impute trừ khi xài Neural Network.
missing_vals = df_features.isnull().sum().sum()
print(f"Tổng số giá trị NaN trong DataFrame: {missing_vals}")

# Hiển thị 5 dòng đầu với max columns
pd.set_option('display.max_columns', 100)
display(df_features.head())

### 5. Lưu Dữ Liệu Dưới Dạng Feather
Lưu vào file `.feather` nhanh hơn nhiều so với `.csv` và `.parquet` khi làm việc với Pandas, đồng thời bảo toàn hoàn toàn Datatype đã ép kiểu.

In [ ]:
PROCESSED_DIR = os.path.join(DATA_DIR, 'processed')
os.makedirs(PROCESSED_DIR, exist_ok=True)

save_path = os.path.join(PROCESSED_DIR, 'features_v2.feather')
print(f"Đang lưu features xuống: {save_path} ...")

# Index phải được reset thì mới lưu feather được
df_features.reset_index(drop=True).to_feather(save_path)

print("Lưu thành công! Sẵn sàng cho Mốc Training Model.")